In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("data/Womens Clothing E-Commerce Reviews.csv")

# Display first five rows
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [2]:
# Remove unnecessary index column
df = df.drop(columns=["Unnamed: 0"])

# Verify
df.head()

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [3]:
# Dataset shape
print("Shape:", df.shape)

# Dataset information
df.info()

Shape: (23486, 10)
<class 'pandas.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Clothing ID              23486 non-null  int64
 1   Age                      23486 non-null  int64
 2   Title                    19676 non-null  str  
 3   Review Text              22641 non-null  str  
 4   Rating                   23486 non-null  int64
 5   Recommended IND          23486 non-null  int64
 6   Positive Feedback Count  23486 non-null  int64
 7   Division Name            23472 non-null  str  
 8   Department Name          23472 non-null  str  
 9   Class Name               23472 non-null  str  
dtypes: int64(5), str(5)
memory usage: 1.8 MB


In [ ]:
# Check missing values
df.isnull().sum()

Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64

In [5]:
# Remove rows with missing review text only
df = df.dropna(subset=["Review Text"])

# Check missing values again
df.isnull().sum()

Clothing ID                   0
Age                           0
Title                      2966
Review Text                   0
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                13
Department Name              13
Class Name                   13
dtype: int64

In [6]:
# Display the first five reviews
df["Review Text"].head()

0    Absolutely wonderful - silky and sexy and comf...
1    Love this dress!  it's sooo pretty.  i happene...
2    I had such high hopes for this dress and reall...
3    I love, love, love this jumpsuit. it's fun, fl...
4    This shirt is very flattering to all due to th...
Name: Review Text, dtype: str

In [7]:
for i, review in enumerate(df["Review Text"].head(), start=1):
    print(f"Review {i}:")
    print(review)
    print("-" * 80)

Review 1:
Absolutely wonderful - silky and sexy and comfortable
--------------------------------------------------------------------------------
Review 2:
Love this dress!  it's sooo pretty.  i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite.  i bought a petite and am 5'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.
--------------------------------------------------------------------------------
Review 3:
I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the n

In [8]:
from prompts.prompt_templates import (
    ZERO_SHOT_PROMPT,
    FEW_SHOT_PROMPT,
    ROLE_PROMPT
)

ASPECT PROMPT LOADED


In [11]:
review = df["Review Text"].iloc[0]

print(review)

Absolutely wonderful - silky and sexy and comfortable


In [9]:
zero_prompt = ZERO_SHOT_PROMPT.format(review=review)

print(zero_prompt)


You are given a customer clothing review.

Classify the overall sentiment of the review.

Respond ONLY in the following JSON format:

{
  "label": "Positive | Negative | Mixed",
  "confidence": "High | Medium | Low",
  "reason": "Brief explanation"
}

Review:
This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!



In [23]:
import google.genai
import dotenv

print("Gemini and dotenv imported successfully!")

Gemini and dotenv imported successfully!


In [12]:
from llm_utils import call_llm

In [13]:
import inspect
print(inspect.getsource(call_llm))

def call_llm(prompt, temperature=0.2, max_tokens=500):
    """
    Sends a prompt to Gemini and retries up to 3 times
    if the API call fails.

    Returns:
        str: Model response if successful.
        None: If all 3 attempts fail.
    """

    api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        raise ValueError(
            "GEMINI_API_KEY not found in environment variables."
        )

    client = genai.Client(api_key=api_key)

    for attempt in range(1, 4):
        try:
            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=prompt,
                config={
                    "temperature": temperature,
                    "max_output_tokens": max_tokens,
                    "response_mime_type": "application/json"
                }
            )

            return response.text

        except Exception as e:
            print(f"Attempt {attempt}/3 failed: {e}")

            if attempt < 3:

In [14]:
response = call_llm(
    zero_prompt,
    temperature=0.2,
    max_tokens=500
)

print(response)

{
  "label": "Positive",
  "confidence": "High",
  "reason": "The reviewer uses strong positive phrases like 'very flattering', 'perfect length', and explicitly states 'love this shirt!!!'."
}


In [15]:
templates = {
    "Zero-Shot": ZERO_SHOT_PROMPT,
    "Few-Shot": FEW_SHOT_PROMPT,
    "Role-Prompted": ROLE_PROMPT
}

print(templates.keys())

dict_keys(['Zero-Shot', 'Few-Shot', 'Role-Prompted'])


In [16]:
reviews = df["Review Text"].head(5).tolist()

print(reviews)

['Absolutely wonderful - silky and sexy and comfortable', 'Love this dress!  it\'s sooo pretty.  i happened to find it in a store, and i\'m glad i did bc i never would have ordered it online bc it\'s petite.  i bought a petite and am 5\'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.', 'I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c', "I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",

In [37]:
import json

results = []

for template_name, template in templates.items():
    for i, review in enumerate(reviews, start=1):

        prompt = template.format(review=review)

        try:
            response = call_llm(
                prompt,
                temperature=0.2,
                max_tokens=500
            )

            parsed = json.loads(response)

            results.append({
                "template": template_name,
                "record": i,
                "review": review,
                "response": parsed,
                "valid_json": True
            })

        except Exception as e:
            print(f"Failed: {template_name} - Record {i}")
            print(f"Error: {e}")

            results.append({
                "template": template_name,
                "record": i,
                "review": review,
                "response": None,
                "valid_json": False
            })

print(f"Total calls attempted: {len(results)}")

Failed: Zero-Shot - Record 2
Error: Expecting value: line 1 column 1 (char 0)
Failed: Zero-Shot - Record 3
Error: Expecting value: line 1 column 1 (char 0)
Failed: Zero-Shot - Record 5
Error: Expecting value: line 1 column 1 (char 0)
Failed: Few-Shot - Record 2
Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 26.603211815s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc

In [20]:
for item in results:
    print(
        item["template"],
        "Record", item["record"],
        "→",
        item["valid_json"]
    )

Zero-Shot Record 1 → True
Zero-Shot Record 2 → False
Zero-Shot Record 3 → False
Zero-Shot Record 4 → True
Zero-Shot Record 5 → True
Few-Shot Record 1 → True
Few-Shot Record 2 → False
Few-Shot Record 3 → False
Few-Shot Record 4 → False
Few-Shot Record 5 → False
Role-Prompted Record 1 → False
Role-Prompted Record 2 → False
Role-Prompted Record 3 → False
Role-Prompted Record 4 → False
Role-Prompted Record 5 → False


In [17]:
import importlib
import prompts.prompt_templates as pt

importlib.reload(pt)

print(hasattr(pt, "ASPECT_PROMPT"))

ASPECT PROMPT LOADED
True


In [18]:
from prompts.prompt_templates import ASPECT_PROMPT

In [19]:
aspect_reviews = df["Review Text"].head(10).tolist()

print(f"Number of reviews: {len(aspect_reviews)}")

for i, review in enumerate(aspect_reviews, start=1):
    print(f"\nReview {i}:")
    print(review)

Number of reviews: 10

Review 1:
Absolutely wonderful - silky and sexy and comfortable

Review 2:
Love this dress!  it's sooo pretty.  i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite.  i bought a petite and am 5'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.

Review 3:
I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c

Review 4:
I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every tim

In [20]:
aspect_results = []

for i, review in enumerate(aspect_reviews, start=1):
    prompt = ASPECT_PROMPT.format(review=review)

    aspect_results.append({
        "record": i,
        "review": review,
        "prompt": prompt
    })

print(f"Prepared {len(aspect_results)} aspect-analysis prompts.")

Prepared 10 aspect-analysis prompts.


In [ ]:
for item in results:
    print(
        item["template"],
        "Record", item["record"],
        "→",
        item["valid_json"]
    )

In [39]:
import importlib
import llm_utils

importlib.reload(llm_utils)

call_llm = llm_utils.call_llm

In [40]:
import inspect

print(inspect.getsource(call_llm))

def call_llm(prompt, temperature=0.2, max_tokens=500):
    """
    Sends a prompt to Gemini and retries up to 3 times
    if the API call fails.
    """

    api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        raise ValueError("GEMINI_API_KEY not found in environment variables.")

    client = genai.Client(api_key=api_key)

    for attempt in range(1, 4):
        try:
            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=prompt,
                config={
                    "temperature": temperature,
                    "max_output_tokens": max_tokens
                }
            )

            return response.text

        except Exception as e:
            print(f"Attempt {attempt}/3 failed: {e}")

            if attempt < 3:
                wait_time = 30 * attempt
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                prin

In [41]:
aspect_outputs = []

for item in aspect_results:
    try:
        response = call_llm(
            item["prompt"],
            temperature=0.2,
            max_tokens=500
        )

        parsed = json.loads(response)

        aspect_outputs.append({
            "record": item["record"],
            "review": item["review"],
            "response": parsed,
            "valid_json": True
        })

        print(f"Record {item['record']} → Success")

    except Exception as e:
        print(f"Record {item['record']} → Failed: {e}")

        aspect_outputs.append({
            "record": item["record"],
            "review": item["review"],
            "response": None,
            "valid_json": False
        })

Record 1 → Failed: Unterminated string starting at: line 3 column 18 (char 42)
Attempt 1/3 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 30 seconds...
Record 2 → Failed: Expecting value: line 1 column 1 (char 0)
Record 3 → Failed: Expecting property name enclosed in double quotes: line 4 column 1 (char 54)
Record 4 → Failed: Unterminated string starting at: line 4 column 5 (char 58)
Record 5 → Failed: Expecting property name enclosed in double quotes: line 3 column 34 (char 58)
Record 6 → Failed: Expecting property name enclosed in double quotes: line 4 column 5 (char 58)
Record 7 → Failed: Expecting value: line 1 column 2 (char 1)
Attempt 1/3 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 's

In [42]:
import importlib
import llm_utils

importlib.reload(llm_utils)

call_llm = llm_utils.call_llm

In [1]:
import pandas as pd
import json

df = pd.read_csv("data/Womens Clothing E-Commerce Reviews.csv")

df = df.drop(columns=["Unnamed: 0"])
df = df.dropna(subset=["Review Text"])

print("Shape:", df.shape)

Shape: (22641, 10)


In [2]:
from prompts.prompt_templates import (
    ZERO_SHOT_PROMPT,
    FEW_SHOT_PROMPT,
    ROLE_PROMPT
)

templates = {
    "Zero-Shot": ZERO_SHOT_PROMPT,
    "Few-Shot": FEW_SHOT_PROMPT,
    "Role-Prompted": ROLE_PROMPT
}

print(templates.keys())

ASPECT PROMPT LOADED
dict_keys(['Zero-Shot', 'Few-Shot', 'Role-Prompted'])


In [ ]:
reviews = df["Review Text"].head(5).tolist()

print(reviews)

['Absolutely wonderful - silky and sexy and comfortable', 'Love this dress!  it\'s sooo pretty.  i happened to find it in a store, and i\'m glad i did bc i never would have ordered it online bc it\'s petite.  i bought a petite and am 5\'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.', 'I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c', "I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",

In [4]:
from prompts.prompt_templates import ASPECT_PROMPT

aspect_reviews = df["Review Text"].head(10).tolist()

aspect_results = []

for i, review in enumerate(aspect_reviews, start=1):
    prompt = ASPECT_PROMPT.format(review=review)

    aspect_results.append({
        "record": i,
        "review": review,
        "prompt": prompt
    })

print(f"Prepared {len(aspect_results)} aspect-analysis prompts.")

Prepared 10 aspect-analysis prompts.


In [5]:
import importlib
import llm_utils

importlib.reload(llm_utils)

call_llm = llm_utils.call_llm

print("call_llm loaded successfully.")

call_llm loaded successfully.


In [ ]:
from llm_utils import call_llm
from prompts.prompt_templates import ZERO_SHOT_PROMPT

test_review = "The dress is beautiful and fits perfectly. The fabric is soft and comfortable."

test_prompt = ZERO_SHOT_PROMPT.format(review=test_review)

response = call_llm(
    test_prompt,
    temperature=0.2,
    max_tokens=500
)

print(response)

In [21]:
from llm_utils import call_llm
from prompts.prompt_templates import ZERO_SHOT_PROMPT

test_review = "The dress is beautiful and fits perfectly. The fabric is soft and comfortable."

test_prompt = ZERO_SHOT_PROMPT.format(review=test_review)

response = call_llm(
    test_prompt,
    temperature=0.2,
    max_tokens=500
)

print(response)

{
  "label": "Positive",
  "confidence": "High",
  "reason": "The review expresses complete satisfaction with the dress's appearance, fit, fabric, and comfort without any negative remarks."
}
